# ML HW3 - Image Classification with ResNet

使用 ResNet50 从头训练 (pretrained=False) 完成 Food-11 分类任务

## 1. 导入依赖

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output

## 2. 设置随机种子（保证复现）

In [ ]:
seed = 1
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")

## 3. 数据路径和 transforms

In [ ]:
base_dir = './food11'

# 训练集数据增强
training_tfm = transforms.Compose([
    transforms.Resize([128, 128]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet 归一化
])

# 验证/测试集预处理
test_tfm = transforms.Compose([
    transforms.Resize([128, 128]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 4. 自定义 Dataset 类

In [ ]:
class FoodDataset(Dataset):
    def __init__(self, path, tfm=test_tfm, files=None):
        super().__init__()
        if files is not None:
            self.files = files
        else:
            self.files = sorted([os.path.join(path, x) for x in os.listdir(path) if x.endswith('.jpg')])
        self.tfm = tfm
        print(f"Dataset path: {self.files[0]}")
    
    def __getitem__(self, idx):
        item = self.files[idx]
        img = Image.open(item)
        img = self.tfm(img)
        try:
            label = int(item.split('/')[-1].split('_')[0])
        except:
            label = -1  # 测试集没有标签
        return img, label
    
    def __len__(self):
        return len(self.files)

## 5. 创建 DataLoader

In [ ]:
batch_size = 64

# 数据集
train_set = FoodDataset(os.path.join(base_dir, 'training'), tfm=training_tfm)
val_set = FoodDataset(os.path.join(base_dir, 'validation'), tfm=test_tfm)
test_set = FoodDataset(os.path.join(base_dir, 'test'), tfm=test_tfm)

# DataLoader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

## 6. 定义 ResNet 模型

使用 `torchvision.models.resnet50(pretrained=False)`，并修改最后一层输出 11 类

In [ ]:
def create_resnet50(num_classes=11):
    """
    创建 ResNet50 模型（从头训练，不使用预训练权重）
    
    Args:
        num_classes: 输出类别数，Food-11 数据集为 11
    """
    # 加载 ResNet50，不使用预训练权重
    model = models.resnet50(pretrained=False)
    
    # 查看原全连接层
    print(f"Original FC layer: {model.fc}")
    
    # 修改最后一层：输入维度 2048，输出维度 11
    in_features = model.fc.in_features  # 2048
    model.fc = nn.Linear(in_features, num_classes)
    
    print(f"Modified FC layer: {model.fc}")
    return model

# 创建模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_resnet50(num_classes=11).to(device)

# 统计模型参数数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 7. 定义损失函数和优化器

In [ ]:
criterion = nn.CrossEntropyLoss()

# Adam 优化器
optimizer = optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-5)

# 学习率调度器：每 30 个 epoch 衰减一次
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

print(f"Optimizer: Adam, lr=0.0003")
print(f"Scheduler: StepLR, step_size=30, gamma=0.1")

## 8. 训练函数

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(loader, desc="Training"):
        images, labels = batch
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # 梯度裁剪
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)
        
        optimizer.step()
        
        # 统计
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def validate(model, loader, criterion, device):
    """验证"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            images, labels = batch
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

## 9. 训练循环

In [ ]:
# 训练参数
num_epochs = 100
early_stop_patience = 10
best_acc = 0
patience_counter = 0

# 保存路径
model_dir = './models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'resnet50_best.pt')

# 记录历史
train_losses, train_accs = [], []
val_losses, val_accs = [], []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)
    
    # 训练
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # 验证
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # 学习率调度
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"Learning Rate: {current_lr:.6f}")
    
    # 保存最佳模型
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), model_path)
        print(f"✓ Saved best model with acc: {best_acc:.4f}")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"✗ No improvement ({patience_counter}/{early_stop_patience})")
    
    # 早停
    if patience_counter >= early_stop_patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break
    
    # 可视化
    if (epoch + 1) % 1 == 0:
        clear_output(wait=True)
        
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.title(f'Loss Curve (Epoch {epoch+1})')
        
        plt.subplot(1, 2, 2)
        plt.plot(train_accs, label='Train Acc')
        plt.plot(val_accs, label='Val Acc')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.title(f'Accuracy Curve (Epoch {epoch+1})')
        
        plt.tight_layout()
        plt.show()

print(f"\nTraining completed! Best validation accuracy: {best_acc:.4f}")

## 10. 加载最佳模型并进行预测

In [ ]:
# 加载最佳模型
model = create_resnet50(num_classes=11).to(device)
model.load_state_dict(torch.load(model_path))
model.eval()
print(f"Loaded best model from {model_path}")

In [ ]:
# 测试集预测
predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        images, _ = batch
        images = images.to(device)
        
        outputs = model(images)
        _, predicted = outputs.max(1)
        predictions.extend(predicted.cpu().numpy())

print(f"Total predictions: {len(predictions)}")

## 11. 生成提交文件

In [ ]:
def pad4(i):
    """将数字补零为4位字符串"""
    return "0" * (4 - len(str(i))) + str(i)

# 创建 DataFrame
df = pd.DataFrame()
df["Id"] = [pad4(i) for i in range(1, len(test_set) + 1)]
df["Category"] = predictions

# 保存为 CSV
submission_path = 'submission_resnet50.csv'
df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(df.head(10))

## 12. 可视化训练结果

In [ ]:
# 最终可视化
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss Curve')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc', linewidth=2)
plt.plot(val_accs, label='Val Acc', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title(f'Accuracy Curve (Best: {best_acc:.4f})')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 总结

本 notebook 使用 **ResNet50** 从头训练 (`pretrained=False`) 完成 Food-11 图像分类任务：

1. **模型**: ResNet50，修改最后一层输出 11 类
2. **输入尺寸**: 128×128
3. **数据增强**: RandomFlip, RandomRotation, ColorJitter, RandomAffine
4. **归一化**: ImageNet mean/std
5. **优化器**: Adam (lr=0.0003) + StepLR 调度
6. **早停**: patience=10

可以在此基础上尝试：
- 更大的输入尺寸（如 224×224）
- 更长的训练时间
- Test Time Augmentation (TTA)
- 模型 Ensemble